# House Prices — Análise Exploratória de Dados (EDA)

**Dataset:** House Prices: Advanced Regression Techniques — Kaggle  
**Objetivo:** Investigar o dataset, testar hipóteses iniciais e identificar padrões que guiem a modelagem.  
**Etapas cobertas neste notebook:**
1. Carregamento e visão geral
2. Análise da variável-alvo (SalePrice)
3. Variáveis numéricas — correlações e distribuições
4. Variáveis categóricas — impacto no preço
5. Análise de valores ausentes
6. Teste das hipóteses
7. Conclusões da EDA

## 0. Instalação e importações

In [ ]:
# Instalar dependências (rodar apenas uma vez)
#!py -m pip install pandas numpy matplotlib seaborn scikit-learn xgboost

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------  9.7/9.9 MB 53.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 47.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ---------------------------------------  12.3/12.5 MB 81.3 MB/s eta 0:00:01
   ---------------------------------------- 12.5/12.5 MB 50.2 MB/s  0:00:00
   ----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('Bibliotecas carregadas com sucesso.')

## 1. Carregamento e visão geral do dataset

In [ ]:
# Ajuste o caminho conforme sua estrutura de pastas
df = pd.read_csv('../data/train.csv')

print(f'Linhas:   {df.shape[0]}')
print(f'Colunas:  {df.shape[1]}')
print(f'Memória:  {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Primeiras linhas
df.head()

In [ ]:
# Tipos de variáveis
tipos = pd.DataFrame({
    'tipo': df.dtypes,
    'nulos': df.isnull().sum(),
    'pct_nulos': (df.isnull().sum() / len(df) * 100).round(1),
    'valores_unicos': df.nunique()
})

print(f"Variáveis numéricas:   {df.select_dtypes(include='number').shape[1]}")
print(f"Variáveis categóricas: {df.select_dtypes(include='object').shape[1]}")
print()
tipos[tipos['nulos'] > 0].sort_values('pct_nulos', ascending=False)

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe().T.sort_values('std', ascending=False).head(20)

## 2. Análise da variável-alvo — SalePrice

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição original
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice — distribuição original')
axes[0].set_xlabel('Preço (USD)')
axes[0].set_ylabel('Frequência')

# Distribuição log-transformada
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('SalePrice — log-transformado')
axes[1].set_xlabel('log(Preço)')
axes[1].set_ylabel('Frequência')

# Boxplot
axes[2].boxplot(df['SalePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('SalePrice — boxplot')
axes[2].set_ylabel('Preço (USD)')

plt.tight_layout()
plt.savefig('../report/fig01_saleprice_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Média:    ${df['SalePrice'].mean():,.0f}")
print(f"Mediana:  ${df['SalePrice'].median():,.0f}")
print(f"Mínimo:   ${df['SalePrice'].min():,.0f}")
print(f"Máximo:   ${df['SalePrice'].max():,.0f}")
print(f"Assimetria (skewness): {df['SalePrice'].skew():.2f}")

## 3. Variáveis numéricas — correlações com SalePrice

Identificar quais variáveis numéricas têm maior relação linear com o preço.

In [ ]:
# Correlação de todas as variáveis numéricas com SalePrice
numericas = df.select_dtypes(include='number').drop(columns=['Id'])
correlacoes = numericas.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

# Top 15 positivas e top 5 negativas
top_corr = pd.concat([correlacoes.head(15), correlacoes.tail(5)])

fig, ax = plt.subplots(figsize=(10, 7))
cores = ['steelblue' if v > 0 else 'salmon' for v in top_corr.values]
ax.barh(top_corr.index[::-1], top_corr.values[::-1], color=cores[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação com SalePrice — top variáveis numéricas')
ax.set_xlabel('Coeficiente de correlação de Pearson')
plt.tight_layout()
plt.savefig('../report/fig02_correlacoes.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 variáveis mais correlacionadas:')
print(correlacoes.head(10).to_string())

In [ ]:
# Heatmap das variáveis mais correlacionadas entre si
top10_vars = correlacoes.head(10).index.tolist() + ['SalePrice']
matriz_corr = df[top10_vars].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(matriz_corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Heatmap de correlação — top 10 variáveis + SalePrice')
plt.tight_layout()
plt.savefig('../report/fig03_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Variáveis categóricas — impacto no preço

In [ ]:
# Preço mediano por bairro
preco_bairro = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
preco_bairro.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Preço mediano por bairro (Neighborhood)')
ax.set_xlabel('Bairro')
ax.set_ylabel('Preço mediano (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../report/fig04_preco_bairro.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição de preço por zona de zoneamento
fig, ax = plt.subplots(figsize=(10, 5))
ordem = df.groupby('MSZoning')['SalePrice'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='MSZoning', y='SalePrice', order=ordem, ax=ax, palette='muted')
ax.set_title('Distribuição de SalePrice por zona (MSZoning)')
ax.set_xlabel('Zona')
ax.set_ylabel('Preço (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('../report/fig05_preco_zona.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise de valores ausentes

In [ ]:
# Variáveis com valores ausentes
ausentes = df.isnull().sum()
ausentes = ausentes[ausentes > 0].sort_values(ascending=False)
pct_ausentes = (ausentes / len(df) * 100).round(1)

resumo_ausentes = pd.DataFrame({
    'nulos': ausentes,
    'percentual': pct_ausentes
})

fig, ax = plt.subplots(figsize=(10, 7))
cores = ['salmon' if p > 50 else 'steelblue' for p in pct_ausentes.values]
ax.barh(ausentes.index[::-1], pct_ausentes.values[::-1], color=cores[::-1])
ax.axvline(50, color='red', linewidth=1, linestyle='--', label='50% de ausência')
ax.set_title('Percentual de valores ausentes por variável')
ax.set_xlabel('% de valores ausentes')
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig06_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

print('Variáveis com mais de 50% de ausência (candidatas à exclusão):')
print(resumo_ausentes[resumo_ausentes['percentual'] > 50])

## 6. Teste das hipóteses

### H1 — A área total (GrLivArea) é o fator mais correlacionado com o preço

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, color='steelblue', edgecolors='white', linewidths=0.3)

# Linha de tendência
z = np.polyfit(df['GrLivArea'], df['SalePrice'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['GrLivArea'].min(), df['GrLivArea'].max(), 200)
ax.plot(x_line, p(x_line), color='salmon', linewidth=2, label='Tendência linear')

# Destacar outliers óbvios
outliers = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000)]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'],
           color='red', zorder=5, label=f'Outliers suspeitos (n={len(outliers)})')

corr_h1 = df['GrLivArea'].corr(df['SalePrice'])
ax.set_title(f'H1: GrLivArea × SalePrice  (r = {corr_h1:.3f})')
ax.set_xlabel('Área total acima do solo (pés²)')
ax.set_ylabel('Preço de venda (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig07_h1_grlivarea.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlação GrLivArea × SalePrice: {corr_h1:.3f}')
print(f'Conclusão H1: {"CONFIRMADA" if corr_h1 > 0.7 else "PARCIALMENTE CONFIRMADA"} — correlação forte, mas presença de outliers.')

### H2 — OverallQual tem impacto não-linear no preço

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot por qualidade
sns.boxplot(data=df, x='OverallQual', y='SalePrice', ax=axes[0], palette='Blues')
axes[0].set_title('H2: SalePrice por qualidade geral (OverallQual)')
axes[0].set_xlabel('Qualidade geral (1=Péssimo, 10=Excelente)')
axes[0].set_ylabel('Preço (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Preço mediano por qualidade
mediana_qualidade = df.groupby('OverallQual')['SalePrice'].median()
axes[1].plot(mediana_qualidade.index, mediana_qualidade.values,
             marker='o', color='steelblue', linewidth=2)
axes[1].set_title('Preço mediano por nível de qualidade')
axes[1].set_xlabel('OverallQual')
axes[1].set_ylabel('Preço mediano (USD)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../report/fig08_h2_qualidade.png', dpi=150, bbox_inches='tight')
plt.show()

# Verificar salto entre qualidade 8, 9 e 10
print('Preço mediano por OverallQual:')
print(mediana_qualidade.apply(lambda x: f'${x:,.0f}').to_string())
print()
print('Conclusão H2: verificar se o salto de qualidade 8 → 10 é desproporcionalmente maior.')

### H3 — O bairro explica variações de preço além do tamanho da casa

In [ ]:
# Comparar casas de área similar em bairros diferentes
area_mediana = df['GrLivArea'].median()
faixa = 200  # +/- 200 pés²
casas_similares = df[
    (df['GrLivArea'] >= area_mediana - faixa) &
    (df['GrLivArea'] <= area_mediana + faixa)
].copy()

preco_bairro_similar = casas_similares.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
preco_bairro_similar.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'H3: Preço mediano por bairro — casas com área similar ({area_mediana-faixa:.0f}–{area_mediana+faixa:.0f} pés², n={len(casas_similares)})')
ax.set_xlabel('Bairro')
ax.set_ylabel('Preço mediano (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../report/fig09_h3_bairro.png', dpi=150, bbox_inches='tight')
plt.show()

variacao = preco_bairro_similar.max() / preco_bairro_similar.min()
print(f'Variação máx/mín entre bairros (área similar): {variacao:.1f}x')
print(f'Conclusão H3: {"CONFIRMADA" if variacao > 1.5 else "NÃO CONFIRMADA"} — bairro explica variações além do tamanho.')

### H4 — Casas sem garagem têm preço sistematicamente menor

In [ ]:
# NaN em GarageType = sem garagem
df['tem_garagem'] = df['GarageType'].notna().map({True: 'Com garagem', False: 'Sem garagem'})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x='tem_garagem', y='SalePrice', ax=axes[0], palette='muted')
axes[0].set_title('H4: SalePrice — com vs. sem garagem')
axes[0].set_xlabel('')
axes[0].set_ylabel('Preço (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Contagem
df['tem_garagem'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'], edgecolor='white')
axes[1].set_title('Distribuição: com vs. sem garagem')
axes[1].set_xlabel('')
axes[1].set_ylabel('Quantidade de imóveis')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('../report/fig10_h4_garagem.png', dpi=150, bbox_inches='tight')
plt.show()

mediana_com = df[df['tem_garagem'] == 'Com garagem']['SalePrice'].median()
mediana_sem = df[df['tem_garagem'] == 'Sem garagem']['SalePrice'].median()
print(f'Preço mediano COM garagem: ${mediana_com:,.0f}')
print(f'Preço mediano SEM garagem: ${mediana_sem:,.0f}')
print(f'Diferença: {((mediana_com - mediana_sem) / mediana_sem * 100):.1f}%')
print(f'Conclusão H4: {"CONFIRMADA" if mediana_com > mediana_sem * 1.1 else "NÃO CONFIRMADA"}')

### H5 — Ano de construção recente está positivamente associado ao preço

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: YearBuilt × SalePrice
axes[0].scatter(df['YearBuilt'], df['SalePrice'], alpha=0.3, color='steelblue', edgecolors='white', linewidths=0.2)
corr_ano = df['YearBuilt'].corr(df['SalePrice'])
axes[0].set_title(f'H5: YearBuilt × SalePrice  (r = {corr_ano:.3f})')
axes[0].set_xlabel('Ano de construção')
axes[0].set_ylabel('Preço (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Preço mediano por década
df['decada'] = (df['YearBuilt'] // 10) * 10
preco_decada = df.groupby('decada')['SalePrice'].median()
axes[1].plot(preco_decada.index, preco_decada.values, marker='o', color='seagreen', linewidth=2)
axes[1].set_title('Preço mediano por década de construção')
axes[1].set_xlabel('Década')
axes[1].set_ylabel('Preço mediano (USD)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../report/fig11_h5_anoconstrucao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlação YearBuilt × SalePrice: {corr_ano:.3f}')
print(f'Conclusão H5: {"CONFIRMADA" if corr_ano > 0.4 else "PARCIALMENTE CONFIRMADA"} — imóveis mais novos tendem a ser mais caros.')

## 7. Conclusões da EDA

In [ ]:
print('=' * 60)
print('RESUMO DAS CONCLUSÕES DA EDA')
print('=' * 60)

conclusoes = {
    'SalePrice':    'Distribuição assimétrica (cauda direita). Usar log1p na modelagem.',
    'H1 (GrLivArea)': 'Confirmada. Maior correlação individual com SalePrice. Atenção aos outliers.',
    'H2 (OverallQual)': 'Confirmada. Relação não-linear — qualidade 9-10 têm prêmio desproporcionalmente alto.',
    'H3 (Neighborhood)': 'Confirmar com output acima. Bairro é variável categórica com alto poder explicativo.',
    'H4 (Garagem)':  'Confirmar com output acima. Presença de garagem adiciona valor significativo.',
    'H5 (YearBuilt)': 'Confirmada. Correlação moderada. Imóveis pós-2000 têm valorização acelerada.',
    'Missing values': 'Colunas com >50% ausentes (PoolQC, MiscFeature, Alley, Fence) serão excluídas ou agrupadas.',
    'Próximos passos': 'Notebook 02: limpeza, encoding de categóricas, feature engineering e normalização.',
}

for chave, valor in conclusoes.items():
    print(f'\n[{chave}]')
    print(f'  {valor}')

In [ ]:
# Salvar dataset com colunas auxiliares criadas durante a EDA
df.to_csv('../data/train_eda.csv', index=False)
print('Dataset salvo em ../data/train_eda.csv')
print('Notebook 01_eda concluído.')